# SIA P1 — Device-Actions LoRA Training (Google Cloud)

Google Cloud version of the Colab notebook. It fine-tunes the SIA
device-actions LoRA adapter using the plain Hugging Face stack
(`transformers` + `peft` + `trl`) via `sia-lab/posttrain/train_gcp.py` —
no unsloth toolchain required.

**Where to run this:** a **Vertex AI Workbench** instance with a GPU
(L4 or T4), or any GCP GPU VM built from a CUDA Deep Learning image with
JupyterLab. Select a **GPU** kernel/runtime before running.

## Step 1 — Check the GPU

In [21]:
!nvidia-smi

Tue Jul  7 18:01:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2 — Get the code + data
Clone the repo and `cd` into it.

In [22]:
import os
work_dir = '/content' if os.path.isdir('/content') else '/home/jupyter' if os.path.isdir('/home/jupyter') else os.getcwd()
os.chdir(work_dir)
!rm -rf SIA
!git clone https://github.com/skmandal3240/SIA.git
os.chdir('SIA')
print('cwd:', os.getcwd())

Cloning into 'SIA'...
remote: Enumerating objects: 387, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 387 (delta 26), reused 23 (delta 20), pack-reused 351 (from 1)
Receiving objects: 100% (387/387), 356.42 KiB | 18.76 MiB/s, done.
Resolving deltas: 100% (195/195), done.
cwd: /content/SIA


## Step 3 — Install the training deps
Deep Learning / Vertex images already ship a CUDA `torch`; the rest are light.

In [23]:
!pip install -q -r sia-lab/posttrain/requirements-train.txt

## Step 4 — (Gated base models only) authenticate to Hugging Face
`meta-llama/*` needs a token + accepted license. Skip this cell if you use
a public mirror such as `unsloth/Llama-3.2-1B-Instruct`.

In [24]:
from huggingface_hub import notebook_login
notebook_login()

## Step 5 — Validate data + config (no GPU spent)

In [25]:
!python3 sia-lab/posttrain/train_gcp.py --dry-run

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
trl import ok
train samples : 1000
val samples   : 100
base model    : unsloth/Llama-3.2-1B-Instruct
LoRA          : r=16 alpha=16 targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
special tokens present in data: ['<|sia_tool|>', '<|sia_call|>', '<|sia_endcall|>', '<|sia_action|>', '<|sia_endaction|>', '<|sia_screen|>']
DRY-RUN OK — data + config validated (no GPU used)


## Step 6 — Train, evaluate, and export the adapter
Set `OUTPUT_DIR` to a `gs://` bucket path to persist the adapter beyond
this instance, or a local path to keep it on disk. `--merge` also writes a
merged fp16 model for serving.

In [20]:
BASE = 'unsloth/Llama-3.2-1B-Instruct'
# Changed to a local path to avoid the GCS Bucket error
OUTPUT_DIR = './outputs/device_actions_lora'

!python3 sia-lab/posttrain/train_gcp.py \
    --base $BASE \
    --train sia-lab/posttrain/data/device_actions_train.json \
    --val   sia-lab/posttrain/data/device_actions_val.json \
    --output-dir $OUTPUT_DIR \
    --epochs 3 --merge

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 146/146 [00:07<00:00, 19.56it/s]
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
added 6 SIA special tokens
trainable params: 11,272,192 || all params: 1,247,098,880 || trainable%: 0.9039
response-only collator unavailable (cannot import name 'DataCollatorForCompletionOnlyLM' from 

## Step 7 — Inspect the held-out accuracy

In [26]:
import json, glob
for p in glob.glob(f'{OUTPUT_DIR}/**/eval.json', recursive=True):
    print(p, json.load(open(p)))

## Step 8 — Download the adapter locally (optional)
If you trained to a local path (not `gs://`), zip and pull it down.

In [27]:
import shutil, os
src = OUTPUT_DIR
if os.path.isdir(src):
    shutil.make_archive('sia_device_actions_lora', 'zip', src)
    print('created sia_device_actions_lora.zip — download it from the file browser')
else:
    print('adapter went to a gs:// bucket; fetch it with: gsutil -m cp -r', OUTPUT_DIR)

adapter went to a gs:// bucket; fetch it with: gsutil -m cp -r 
